In [1]:
import time

import pandas as pd

from prompt_utils import get_prompt, print_response, print_local_response
from data_utils import batch_df, parse_predictions, create_comparison_df, read_dataset

In [2]:
english_train_df = read_dataset("../data/dev_phase/subtask1/train/eng.csv")

# Gemini

## Batching

In [52]:
english_train_df = english_train_df.sample(frac=1, random_state=4).reset_index(drop=True)

In [53]:
from prompt_utils import get_prompt, print_response, get_gepa_prompt
from data_utils import batch_df, parse_predictions, create_comparison_df

In [54]:
gen = batch_df(english_train_df)
first_batch = next(gen)
gen_list = list(gen)

In [18]:
len(gen_list)

26

In [56]:
def batch_pipeline(batch):
    statements = list(batch["text"])

    prompt = get_gepa_prompt(statements)
    result = print_response(prompt)
    predicted = parse_predictions(result)
    gt = batch["polarization"]
    comparison_df = create_comparison_df(predicted, gt)

    accuracy = (comparison_df["Ground Truth"] == comparison_df["Predicted"]).mean()
    return accuracy, comparison_df

In [ ]:
accuracy, comparison_df = batch_pipeline(first_batch)

In [19]:
type(comparison_df["Predicted"].to_list())

list

In [20]:
y_pred = comparison_df["Predicted"].to_list()
y_true = comparison_df["Ground Truth"].to_list()

In [42]:
predictions_list = [(yt, yp) for yt, yp in zip(y_true, y_pred)]

In [46]:
tp = sum((yt == 1 and yp == 1) for yt, yp in zip(y_true, y_pred))
fp = sum((yt == 0 and yp == 1) for yt, yp in zip(y_true, y_pred))
fn = sum((yt == 1 and yp == 0) for yt, yp in zip(y_true, y_pred))

In [49]:
import numpy as np
print(np.unique(y_pred, return_counts=True))

(array([0]), array([100]))


In [47]:
print(tp, fp, fn)

0 0 32


In [30]:
from sklearn.metrics import f1_score, accuracy_score

accuracy_score = accuracy_score(y_true, y_pred)

0.0


In [11]:
print(batch_pipeline(first_batch))

0.68


In [12]:
print(batch_pipeline(gen_list[1]))

0.5918367346938775


In [13]:
print(batch_pipeline(gen_list[2]))

0.64


In [14]:
print(batch_pipeline(gen_list[3]))

0.6


In [30]:
print(batch_pipeline(gen_list[4]))

0.87


In [24]:
import time